# 01 · Bronze ingestion
Reads the raw CSV from the landing Volume and writes it to Delta with an explicit schema (the source file has two columns named `Account`) and audit columns. No business logic at this layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

CAT = "workspace.aml_lakehouse"
raw_path = "/Volumes/workspace/aml_lakehouse/raw/HI-Small_Trans.csv"

schema = StructType([
    StructField("timestamp",          StringType(), True),
    StructField("from_bank",          StringType(), True),
    StructField("from_account",       StringType(), True),
    StructField("to_bank",            StringType(), True),
    StructField("to_account",         StringType(), True),
    StructField("amount_received",    DoubleType(), True),
    StructField("receiving_currency", StringType(), True),
    StructField("amount_paid",        DoubleType(), True),
    StructField("payment_currency",   StringType(), True),
    StructField("payment_format",     StringType(), True),
    StructField("is_laundering",      IntegerType(), True),
])

bronze = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(raw_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

bronze.write.format("delta").mode("overwrite").saveAsTable(f"{CAT}.bronze_transactions")

In [0]:
df = spark.table(f"{CAT}.bronze_transactions")
print("rows:", df.count())
display(df.groupBy("is_laundering").count())